In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/time-series-air-quality-data-of-india-2010-2023/DL023.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/WB005.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/MH018.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/UP017.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/BR008.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/BR003.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/MP016.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/KA035.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/CH001.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/PY001.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/DL040.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/OR002.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/UP057.csv
/kaggle/input/time-series-air-quality-data-of-india-2010-2023/KL

In [2]:
# Inventory & metadata generation for the Kaggle dataset
# Run this in a Kaggle notebook with the dataset mounted under /kaggle/input/...

import pandas as pd
import glob
import os
from tqdm import tqdm

DATA_DIR = "/kaggle/input/time-series-air-quality-data-of-india-2010-2023"
OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

# load stations_info if present (helps map file->city/state)
stations_info_path = os.path.join(DATA_DIR, "stations_info.csv")
if os.path.exists(stations_info_path):
    stations_info = pd.read_csv(stations_info_path)
    # normalize file names in stations_info if necessary
    if 'file_name' in stations_info.columns:
        stations_info['file_name'] = stations_info['file_name'].astype(str).str.strip()
else:
    stations_info = None

# find CSV files
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
# filter out stations_info itself
csv_files = [p for p in csv_files if os.path.basename(p).lower() != "stations_info.csv"]

rows = []
# helper to parse datetime robustly
def try_parse_datetime(s):
    try:
        return pd.to_datetime(s, infer_datetime_format=True, errors='coerce')
    except Exception:
        return pd.to_datetime(s, errors='coerce')

for f in tqdm(csv_files, desc="Scanning CSVs"):
    fname = os.path.basename(f)
    try:
        # read only a sample first to inspect columns (faster)
        sample = pd.read_csv(f, nrows=200)
    except Exception as e:
        rows.append({
            "file_name": fname,
            "ok": False,
            "error": f"read_error: {e}"
        })
        continue

    cols = list(sample.columns)
    # try to find datetime column
    dt_col = None
    for c in cols:
        if c.lower() in ("datetime", "date_time", "timestamp", "time", "date"):
            dt_col = c
            break
    # if not obvious, look for a column with many parseable datetimes in sample
    if dt_col is None:
        for c in cols:
            parsed = try_parse_datetime(sample[c])
            if parsed.notna().sum() > len(sample) * 0.5:
                dt_col = c
                break

    # load full csv but only needed cols to reduce memory: dt_col + pollutants
    need_cols = cols.copy()
    usecols = None
    if dt_col:
        usecols = list(set([dt_col] + [c for c in cols if c.lower() in ("pm2.5","pm2.5 (ug/m3)","pm10","pm10 (ug/m3)","pm2.5 (ug/m3)","pm10 (ug/m3)","pm2_5","pm10_ug/m3")]))
        # ensure dt_col present
        if dt_col not in usecols:
            usecols.append(dt_col)
    try:
        if usecols:
            df = pd.read_csv(f, usecols=lambda x: True)  # read all to be safe given variable col names
        else:
            df = pd.read_csv(f)
    except Exception as e:
        rows.append({
            "file_name": fname,
            "ok": False,
            "error": f"full_read_error: {e}"
        })
        continue

    # identify datetime column again if necessary
    if dt_col is None:
        for c in df.columns:
            parsed = try_parse_datetime(df[c].astype(str).fillna(''))
            if parsed.notna().sum() > len(df) * 0.5:
                dt_col = c
                break

    # parse datetimes if found
    if dt_col:
        df['_parsed_dt'] = try_parse_datetime(df[dt_col].astype(str))
        first_ts = df['_parsed_dt'].min()
        last_ts  = df['_parsed_dt'].max()
        n_missing_dt = df['_parsed_dt'].isna().sum()
    else:
        first_ts = None
        last_ts = None
        n_missing_dt = None

    # basic counts
    total_rows = len(df)
    # PM columns detection (common names)
    def find_col(candidates):
        for cand in candidates:
            for col in df.columns:
                if col.lower().replace(" ", "").replace("_","") == cand.lower().replace(" ","").replace("_",""):
                    return col
                if cand.lower() in col.lower():
                    return col
        return None

    pm25_col = find_col(["pm2.5","pm2_5","pm25"])
    pm10_col = find_col(["pm10"])

    pm25_missing_pct = None
    pm10_missing_pct = None
    if pm25_col:
        pm25_missing_pct = df[pm25_col].isna().mean() * 100
    if pm10_col:
        pm10_missing_pct = df[pm10_col].isna().mean() * 100

    # try to map to stations_info
    state = city = agency = None
    if stations_info is not None:
        # station file name might be in file_name or file column without extension
        lookup_name = fname
        match = stations_info[stations_info['file_name'].astype(str).str.contains(os.path.splitext(lookup_name)[0], na=False)]
        if match.shape[0] == 1:
            rowinfo = match.iloc[0]
            state = rowinfo.get('state', None)
            city = rowinfo.get('city', None)
            agency = rowinfo.get('agency', None)

    rows.append({
        "file_name": fname,
        "ok": True,
        "state": state,
        "city": city,
        "agency": agency,
        "n_rows": total_rows,
        "first_timestamp": str(first_ts) if pd.notna(first_ts) else None,
        "last_timestamp": str(last_ts) if pd.notna(last_ts) else None,
        "n_missing_datetime": int(n_missing_dt) if n_missing_dt is not None else None,
        "pm25_col": pm25_col,
        "pm25_missing_pct": round(float(pm25_missing_pct), 2) if pm25_missing_pct is not None else None,
        "pm10_col": pm10_col,
        "pm10_missing_pct": round(float(pm10_missing_pct), 2) if pm10_missing_pct is not None else None,
        "columns": "|".join(df.columns.astype(str).tolist())
    })

# save inventory dataframe
inv = pd.DataFrame(rows)
inv_path = os.path.join(OUT_DIR, "stations_inventory.csv")
inv.to_csv(inv_path, index=False)
print("Wrote inventory to:", inv_path)

# show top 20 rows for quick preview
display(inv.head(20))


Scanning CSVs:   0%|          | 0/453 [00:00<?, ?it/s]/tmp/ipykernel_19/1954205569.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, infer_datetime_format=True, errors='coerce')
/tmp/ipykernel_19/1954205569.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(s, infer_datetime_format=True, errors='coerce')
Scanning CSVs:   0%|          | 1/453 [00:00<02:07,  3.55it/s]/tmp/ipykernel_19/1954205569.py:32: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A

Wrote inventory to: /kaggle/working/stations_inventory.csv


,file_name,ok,state,city,agency,n_rows,first_timestamp,last_timestamp,n_missing_datetime,pm25_col,pm25_missing_pct,pm10_col,pm10_missing_pct,columns
0,AP001.csv,True,Andhra Pradesh,Tirupati,APPCB,59150,2016-07-01 10:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),8.16,PM10 (ug/m3),7.95,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
1,AP002.csv,True,Andhra Pradesh,Vijayawada,APPCB,51864,2017-05-01 00:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),72.64,PM10 (ug/m3),68.98,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
2,AP003.csv,True,Andhra Pradesh,Visakhapatnam,APPCB,50400,2017-07-01 00:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),11.33,PM10 (ug/m3),10.68,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
3,AP004.csv,True,Andhra Pradesh,Rajamahendravaram,APPCB,48802,2017-09-05 14:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),5.04,PM10 (ug/m3),3.86,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
4,AP005.csv,True,Andhra Pradesh,Amaravati,APPCB,46880,2017-11-24 16:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),9.64,PM10 (ug/m3),8.68,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
5,AP006.csv,True,Andhra Pradesh,Anantapur,APPCB,5432,2022-08-17 16:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),4.66,PM10 (ug/m3),4.64,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
6,AP007.csv,True,Andhra Pradesh,Chittoor,APPCB,3127,2022-11-21 17:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),13.11,PM10 (ug/m3),13.18,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
7,AP008.csv,True,Andhra Pradesh,Tirupati,APPCB,3280,2022-11-15 08:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),9.18,PM10 (ug/m3),16.43,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
8,AP009.csv,True,Andhra Pradesh,Vijayawada,APPCB,1689,2023-01-20 15:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),26.35,PM10 (ug/m3),26.41,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...
9,AP010.csv,True,Andhra Pradesh,Kadapa,APPCB,1593,2023-01-24 15:00:00,2023-03-31 23:00:00,0,PM2.5 (ug/m3),15.38,PM10 (ug/m3),15.44,From Date|To Date|PM2.5 (ug/m3)|PM10 (ug/m3)|N...


In [3]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm

DATA_DIR = "/kaggle/input/time-series-air-quality-data-of-india-2010-2023"
stations_info_path = os.path.join(DATA_DIR, "stations_info.csv")

# Load station info
stations_info = pd.read_csv(stations_info_path)
stations_info['file_name'] = stations_info['file_name'].astype(str).str.strip()

# Pick columns we care about (you can add more later)
KEY_COLS = [
    'PM2.5 (ug/m3)', 'PM10 (ug/m3)', 'NO (ug/m3)', 'NO2 (ug/m3)',
    'NOx (ppb)', 'NH3 (ug/m3)', 'SO2 (ug/m3)', 'CO (mg/m3)',
    'Ozone', 'Temp', 'AT', 'RH'
]

# Helper to find datetime column
def detect_datetime_col(df):
    for c in df.columns:
        if c.lower() in ('datetime', 'date_time', 'from date', 'to date', 'time', 'date'):
            return c
    for c in df.columns:
        parsed = pd.to_datetime(df[c], errors='coerce')
        if parsed.notna().sum() > len(df) * 0.6:
            return c
    return None

# Loop through all station files
all_dfs = []
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
files = [f for f in files if "stations_info" not in f.lower()]

for file in tqdm(files[:]):  # can slice [:50] for testing
    try:
        df = pd.read_csv(file)
        dt_col = detect_datetime_col(df)
        if dt_col is None:
            continue

        df['datetime'] = pd.to_datetime(df[dt_col], errors='coerce')
        df = df.dropna(subset=['datetime'])

        # subset columns
        keep_cols = [c for c in df.columns if c in KEY_COLS]
        df = df[['datetime'] + keep_cols].copy()

        # add file_name and merge metadata
        fname = os.path.basename(file)
        meta = stations_info[stations_info['file_name'].str.contains(fname.split('.')[0], case=False, na=False)]
        if not meta.empty:
            df['state'] = meta.iloc[0]['state']
            df['city'] = meta.iloc[0]['city']
            df['station_id'] = fname.split('.')[0]
        else:
            df['state'] = None
            df['city'] = None
            df['station_id'] = fname.split('.')[0]

        all_dfs.append(df)
    except Exception as e:
        print(f"❌ Error reading {file}: {e}")

# Concatenate all
unified_df = pd.concat(all_dfs, ignore_index=True)
print(f"✅ Unified dataset shape: {unified_df.shape}")
unified_df.head()


100%|██████████| 453/453 [00:37<00:00, 12.00it/s]


✅ Unified dataset shape: (14295663, 12)


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,datetime,PM2.5 (ug/m3),PM10 (ug/m3),NO (ug/m3),NO2 (ug/m3),NOx (ppb),NH3 (ug/m3),SO2 (ug/m3),CO (mg/m3),state,city,station_id
0,2016-07-01 10:00:00,10.67,39.0,17.67,39.2,32.33,7.07,6.60,0.48,Andhra Pradesh,Tirupati,AP001
1,2016-07-01 11:00:00,2.00,39.0,20.50,41.9,35.80,7.40,NaN,0.49,Andhra Pradesh,Tirupati,AP001
2,2016-07-01 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Andhra Pradesh,Tirupati,AP001
3,2016-07-01 13:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Andhra Pradesh,Tirupati,AP001
4,2016-07-01 14:00:00,20.50,50.0,15.40,43.6,32.78,6.35,6.38,0.47,Andhra Pradesh,Tirupati,AP001


In [4]:
import pandas as pd
import numpy as np

df = unified_df.copy()

# --- Handle missing & outliers ---
for col in ['PM2.5 (ug/m3)', 'PM10 (ug/m3)', 'NO2 (ug/m3)', 'SO2 (ug/m3)', 'CO (mg/m3)', 'NH3 (ug/m3)']:
    if col in df.columns:
        df[col] = df[col].clip(lower=0)
        upper = df[col].quantile(0.999)
        df[col] = df[col].clip(upper=upper)

# Fill missing by station median
df = df.groupby('station_id').apply(lambda x: x.fillna(x.median(numeric_only=True))).reset_index(drop=True)

# --- Standardize names ---
df.rename(columns={
    'PM2.5 (ug/m3)': 'PM2_5',
    'PM10 (ug/m3)': 'PM10',
    'NO (ug/m3)': 'NO',
    'NO2 (ug/m3)': 'NO2',
    'NOx (ppb)': 'NOx',
    'NH3 (ug/m3)': 'NH3',
    'SO2 (ug/m3)': 'SO2',
    'CO (mg/m3)': 'CO'
}, inplace=True)

# --- Datetime features ---
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = df.dropna(subset=['datetime'])
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)

# --- Season feature ---
def get_season(month):
    if month in [12,1,2]: return "Winter"
    elif month in [3,4,5]: return "Summer"
    elif month in [6,7,8,9]: return "Monsoon"
    else: return "Autumn"

df['season'] = df['month'].apply(get_season)

# --- Composite key ---
df['state_city'] = df['state'].astype(str) + "_" + df['city'].astype(str)

print("✅ Normalized shape:", df.shape)
df.head()


/tmp/ipykernel_19/3970924482.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('station_id').apply(lambda x: x.fillna(x.median(numeric_only=True))).reset_index(drop=True)


✅ Normalized shape: (14295663, 20)


,datetime,PM2_5,PM10,NO,NO2,NOx,NH3,SO2,CO,state,city,station_id,year,month,day,hour,day_of_week,is_weekend,season,state_city
0,2016-07-01 10:00:00,10.67,39.0,17.67,39.20,32.33,7.07,6.60,0.48,Andhra Pradesh,Tirupati,AP001,2016,7,1,10,4,0,Monsoon,Andhra Pradesh_Tirupati
1,2016-07-01 11:00:00,2.00,39.0,20.50,41.90,35.80,7.40,5.15,0.49,Andhra Pradesh,Tirupati,AP001,2016,7,1,11,4,0,Monsoon,Andhra Pradesh_Tirupati
2,2016-07-01 12:00:00,24.00,53.5,8.10,30.65,24.88,8.42,5.15,0.57,Andhra Pradesh,Tirupati,AP001,2016,7,1,12,4,0,Monsoon,Andhra Pradesh_Tirupati
3,2016-07-01 13:00:00,24.00,53.5,8.10,30.65,24.88,8.42,5.15,0.57,Andhra Pradesh,Tirupati,AP001,2016,7,1,13,4,0,Monsoon,Andhra Pradesh_Tirupati
4,2016-07-01 14:00:00,20.50,50.0,15.40,43.60,32.78,6.35,6.38,0.47,Andhra Pradesh,Tirupati,AP001,2016,7,1,14,4,0,Monsoon,Andhra Pradesh_Tirupati


In [5]:
# safer group-wise median fill (avoids future pandas deprecation)
num_cols = df.select_dtypes(include='number').columns.tolist()
# exclude grouping key if present in num_cols
num_cols = [c for c in num_cols if c != 'station_id']

medians = df.groupby('station_id')[num_cols].median()
# join medians back to df and fill
df = df.set_index('station_id')
df[num_cols] = df[num_cols].fillna(medians)
df = df.reset_index()


In [6]:
# ensure datetime is datetime
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

# create hourly -> daily
df['date'] = df['datetime'].dt.date

# daily aggregated averages per station (useful for many models)
daily_station = df.groupby(['station_id','state','city','date']).agg({
    'PM2_5': 'mean',
    'PM10'  : 'mean',
    'NO'    : 'mean' if 'NO' in df.columns else 'first',
    'NO2'   : 'mean' if 'NO2' in df.columns else 'first',
    'SO2'   : 'mean' if 'SO2' in df.columns else 'first',
    'CO'    : 'mean' if 'CO' in df.columns else 'first',
    'NH3'   : 'mean' if 'NH3' in df.columns else 'first'
}).reset_index()

daily_station.rename(columns={'date':'ds'}, inplace=True)  # ds common for forecasting
print("daily_station shape:", daily_station.shape)
daily_station.head()


daily_station shape: (595882, 11)


,station_id,state,city,ds,PM2_5,PM10,NO,NO2,SO2,CO,NH3
0,AP001,Andhra Pradesh,Tirupati,2016-07-01,13.917143,51.178571,13.537857,37.386429,6.053571,0.437857,7.840714
1,AP001,Andhra Pradesh,Tirupati,2016-07-02,22.923750,62.399167,19.819167,40.761667,4.979167,0.906667,7.882083
2,AP001,Andhra Pradesh,Tirupati,2016-07-03,14.162500,46.146250,12.116250,41.458750,5.541250,0.763333,8.693750
3,AP001,Andhra Pradesh,Tirupati,2016-07-04,9.625000,30.267500,7.512500,37.402083,5.696250,0.631250,7.383333
4,AP001,Andhra Pradesh,Tirupati,2016-07-05,4.718750,17.228750,15.559167,39.981667,5.075833,0.752083,7.469583


In [7]:
import pandas as pd
import numpy as np

print("📦 Unified Dataset Overview")
print("-" * 60)
print(f"Total rows: {len(df):,}")
print(f"Total columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print("\n--- Column Info ---")
df.info()

print("\n--- Unique Counts ---")
print(f"States: {df['state'].nunique()}")
print(f"Cities: {df['city'].nunique()}")
print(f"Stations: {df['station_id'].nunique()}")

print("\n--- Datetime Range ---")
print(f"From: {df['datetime'].min()}   →   To: {df['datetime'].max()}")

print("\n--- Missing Values (%) ---")
missing = df.isnull().mean().sort_values(ascending=False) * 100
display(missing.round(2).to_frame("Missing (%)").T)

print("\n--- Pollutant Summary (numeric only) ---")
pollutants = ['PM2_5','PM10','NO','NO2','NOx','NH3','SO2','CO']
desc = df[pollutants].describe(percentiles=[.05,.25,.5,.75,.95]).T
display(desc)

print("\n--- Example Records ---")
display(df.head(10))


📦 Unified Dataset Overview
------------------------------------------------------------
Total rows: 14,295,663
Total columns: 21
Memory usage: 6397.75 MB

--- Column Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14295663 entries, 0 to 14295662
Data columns (total 21 columns):
 #   Column       Dtype         
---  ------       -----         
 0   station_id   object        
 1   datetime     datetime64[ns]
 2   PM2_5        float64       
 3   PM10         float64       
 4   NO           float64       
 5   NO2          float64       
 6   NOx          float64       
 7   NH3          float64       
 8   SO2          float64       
 9   CO           float64       
 10  state        object        
 11  city         object        
 12  year         int32         
 13  month        int32         
 14  day          int32         
 15  hour         int32         
 16  day_of_week  int32         
 17  is_weekend   int64         
 18  season       object        
 19  state_city  

,NH3,SO2,NOx,PM10,CO,NO,PM2_5,NO2,station_id,datetime,...,city,year,month,day,hour,day_of_week,is_weekend,season,state_city,date
Missing (%),22.66,8.28,5.82,4.34,3.86,3.64,1.52,0.16,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



--- Pollutant Summary (numeric only) ---


,count,mean,std,min,5%,25%,50%,75%,95%,max
PM2_5,14077824.0,64.464535,65.329033,0.01,10.13,26.78,47.00,78.58,180.49,759.00000
PM10,13675224.0,130.122604,105.605269,0.01,26.00,60.59,103.67,171.45,326.75,917.00000
NO,13775564.0,16.176937,35.561624,0.01,1.01,3.15,6.91,14.07,60.53,500.00000
NO2,14273448.0,28.097702,28.437266,0.01,3.49,11.67,19.72,33.92,79.88,302.02000
NOx,13463145.0,33.447674,42.721555,0.00,4.07,12.57,21.98,36.57,103.18,500.00000
NH3,11055672.0,25.909466,25.572426,0.01,3.08,10.35,19.02,34.35,67.10,304.56834
SO2,13112299.0,12.428679,14.377721,0.01,1.52,4.75,8.49,15.15,34.37,168.73000
CO,13744264.0,1.027786,1.357832,0.00,0.11,0.47,0.79,1.21,2.49,26.01000



--- Example Records ---


,station_id,datetime,PM2_5,PM10,NO,NO2,NOx,NH3,SO2,CO,...,city,year,month,day,hour,day_of_week,is_weekend,season,state_city,date
0,AP001,2016-07-01 10:00:00,10.67,39.00,17.67,39.20,32.33,7.07,6.60,0.48,...,Tirupati,2016,7,1,10,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
1,AP001,2016-07-01 11:00:00,2.00,39.00,20.50,41.90,35.80,7.40,5.15,0.49,...,Tirupati,2016,7,1,11,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
2,AP001,2016-07-01 12:00:00,24.00,53.50,8.10,30.65,24.88,8.42,5.15,0.57,...,Tirupati,2016,7,1,12,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
3,AP001,2016-07-01 13:00:00,24.00,53.50,8.10,30.65,24.88,8.42,5.15,0.57,...,Tirupati,2016,7,1,13,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
4,AP001,2016-07-01 14:00:00,20.50,50.00,15.40,43.60,32.78,6.35,6.38,0.47,...,Tirupati,2016,7,1,14,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
5,AP001,2016-07-01 15:00:00,15.25,59.50,24.30,45.12,40.12,6.65,6.53,0.51,...,Tirupati,2016,7,1,15,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
6,AP001,2016-07-01 16:00:00,11.67,60.00,26.73,49.10,43.80,6.93,5.70,0.46,...,Tirupati,2016,7,1,16,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
7,AP001,2016-07-01 17:00:00,11.75,57.50,19.10,46.33,36.82,8.12,6.23,0.44,...,Tirupati,2016,7,1,17,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
8,AP001,2016-07-01 18:00:00,18.00,57.75,14.50,44.20,32.33,7.78,6.07,0.45,...,Tirupati,2016,7,1,18,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01
9,AP001,2016-07-01 19:00:00,12.00,63.00,10.43,41.00,27.83,8.13,6.20,0.41,...,Tirupati,2016,7,1,19,4,0,Monsoon,Andhra Pradesh_Tirupati,2016-07-01


In [8]:
# Path to save unified dataset
out_path = "/kaggle/working/unified_air_quality.csv"

# Assuming your final DataFrame is named df
df.to_csv(out_path, index=False)

# Confirm it’s saved
!ls -lh /kaggle/working/


total 1.9G
---------- 1 root root 576K Oct 29 14:22 __notebook__.ipynb
-rw-r--r-- 1 root root 180K Oct 29 14:20 stations_inventory.csv
-rw-r--r-- 1 root root 1.9G Oct 29 14:24 unified_air_quality.csv
